<a href="https://colab.research.google.com/github/dsb4k8/Machine-Learning/blob/main/Subset_Selection_(Resampling)_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross Validation and the Bootstrap
_Cross-validation and the bootstrap are two of the most powerful resampling techniques used to assess a model's performance and quantify the uncertainty of its estimates without requiring a completely new dataset._


In [ ]:
##Grabbing the ISLR library: https://cran.r-project.org/web/packages/ISLR2/index.html
install.packages("ISLR")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



##Validation set approach
Validation sets can be used to estimate test error rated resulting from fitting various linear models (lm) on - in this case - automotive data


For random sampling, RNG is required - but RNG alone introduces problems for reproducability.

For this reason, we first set a fixed `seed value` such that your machine produces the same random values as mine.



In [ ]:
library(ISLR)
library(glue)
set.seed(7829)

##Splitting the data
The `sample()` method is used for splitting the data into 2 subsets.
- First,
generating a dataset of numbers from [1 - 392],
- then,
taking 192 if them at random and assigning them to the variable train.
This variable stores our training set

_for docs, run `?sample` in a separate cell._

In [ ]:
train_50 <-sample(392, 196)

For the sake of comparison, lets test a few other splits:
we already have a 50/50 training. lets also do:

- 60%, 70%, and 70%

test percentage splits

In [ ]:
392 * 0.5
392 * 0.6
392 * .7
392 * .8

[1] 196

[1] 235.2

[1] 274.4

[1] 313.6

####Splitting the data > Quality of Life Update

Since we are working with 4 configuations, lets make a function which wraps the core logic - allowing us just to pass the appropiate parameters

In [ ]:
test_train_split <- function(total_pop, trainPercentage) {
  return (sample(total_pop, total_pop*trainPercentage))
}
#Resetting the seed for each split to ensure we are comparing apples to apples
set.seed(7829)
train_60 <- test_train_split(392, 0.60) # 235
set.seed(7829)
train_70 <- test_train_split(392, 0.70) # 274
set.seed(7829)
train_80 <- test_train_split(392, 0.80) # 313


##Running Linear Regression on the training sets
We now have 4 lists of 4 sample sizes:


`train_60` contains a list of 235 randomly sampled numbers taken from the original set  (1 - 392).

We use this list of random integers as indicies against the cohort of 392.

Then we fit the data using linear regression - comparing mpg against horsepower using the subset of data we sampled.

In [ ]:
lm.fit50 <- lm(mpg ~ horsepower, data = Auto, subset = train_50)
lm.fit60 <- lm(mpg ~ horsepower, data = Auto, subset = train_60)
lm.fit70 <- lm(mpg ~ horsepower, data = Auto, subset = train_70)
lm.fit80 <- lm(mpg ~ horsepower, data = Auto, subset = train_80)

In [41]:
attach(Auto)

The following objects are masked from Auto (pos = 3):

    acceleration, cylinders, displacement, horsepower, mpg, name,
    origin, weight, year




`predict()` estimates the response for all observations

`mean()` will calculate the MSE for all 196 observations in the validation set.

`-train`: is the syntax to use when we only want to select observations which are NOT in the training set

Below, im simply calculating MSE using the y (actual mpg), and y-hat (predicted)

In [42]:
m1 <- mean((mpg - predict(lm.fit50, Auto))[-train_50]^2)
glue("MSE with sample val size 196: {m1}")

m2 <- mean((mpg - predict(lm.fit60, Auto))[-train_60]^2)
glue("MSE with sample size 235: {m2}")

m3 <- mean((mpg - predict(lm.fit70, Auto))[-train_70]^2)
glue("MSE with sample size 274: {m3}")

m4 <- mean((mpg - predict(lm.fit80, Auto))[-train_80]^2)
glue("MSE with sample size 313: {m4}")



MSE with sample val size 196: 24.2771653209809

MSE with sample size 235: 22.9379588335834

MSE with sample size 274: 18.3063977022852

MSE with sample size 313: 19.6861149582437

As we can see,
- with a sample size of 196, Linear regression against mpg using horsepower as our predictor yielded a mean squared error of 24.27.
- with a sample size of 235, Linear regression against mpg using horsepower as our predictor yielded a mean squared error of 22.93
- with a sample size of 274, Linear regression against mpg using horsepower as our predictor yielded a mean squared error of 18.20.
- with a sample size of 313, Linear regression against mpg using horsepower as our predictor yielded a mean squared error of 19.6.

These will be our baseline values for each sample size.

**Next,** lets compare this linear validation performance with quadratic and cubic models for each sample size

Below, `mpg ~ poly(horsepower, 2)` tells R to predict
`mpg` based on
  - `horsepower` &
 - `horsepower^2`

 This represents the quadratic model,

 and `mpg ~ poly(horsepower, 3)` represents the cubic model

In [55]:
glue("__Original_Linear_Regression__")

glue("n=196 | Linear | MSE = 24.2771653209809")
glue("n=235 | Linear | MSE = 29.4771353987652")
glue("n=274 | Linear | MSE = 25.3490811767125")
glue("n=313 | Linear | MSE = 27.2375142172392")

#Building a function around the fits to keep the 8 unique exponent and trainingset combinations from getting cluttered
fit_nonlinear <- function(predict_value, predictor, exponent, training_set) {
  lm.fit <- lm(predict_value ~ poly(predictor, exponent), data = Auto, subset = training_set)
  avrg <- mean((predict_value - predict(lm.fit, Auto))[-training_set]^2)
  glue("n={length(training_set)} | {ifelse(exponent == 2, 'Quadratic', ifelse(exponent == 3, 'Cubic', 'Linear'))} | MSE = {avrg}")
  # return(avrg)
}


glue("______Quadriatics______")
#Quadriatics for each sample size
fit_nonlinear(mpg, horsepower, 2, train_50)
fit_nonlinear(mpg, horsepower, 2, train_60)
fit_nonlinear(mpg, horsepower, 2, train_70)
fit_nonlinear(mpg, horsepower, 2, train_80)

glue("_________Cubic_________")
#Cubics for each sample size
fit_nonlinear(mpg, horsepower, 3, train_50)
fit_nonlinear(mpg, horsepower, 3, train_60)
fit_nonlinear(mpg, horsepower, 3, train_70)
fit_nonlinear(mpg, horsepower, 3, train_80)


__Original_Linear_Regression__

n=196 | Linear | MSE = 24.2771653209809

n=235 | Linear | MSE = 29.4771353987652

n=274 | Linear | MSE = 25.3490811767125

n=313 | Linear | MSE = 27.2375142172392

______Quadriatics______

... (5 outputs left out)

n=196 | Cubic | MSE = 18.3951194207187

n=235 | Cubic | MSE = 18.3635286782672

n=274 | Cubic | MSE = 14.5812836530113

n=313 | Cubic | MSE = 16.4044516209257

As we can see, going from a linear model to a quadratic model has the most dramatic change, while going from quadratic to cubic only marginally improves the error rate.